# Stratifying consumer data


In [4]:
import Pkg; Pkg.add("DataFrames"); Pkg.add("JSON")
using DataFrames
using JSON


   Resolving package versions...
      Compat entries added for 
     Project No packages added to or removed from `~/CODE/OpenDleto/Project.toml`
    Manifest No packages added to or removed from `~/CODE/OpenDleto/Manifest.toml`
   Resolving package versions...
      Compat entries added for 
     Project No packages added to or removed from `~/CODE/OpenDleto/Project.toml`
    Manifest No packages added to or removed from `~/CODE/OpenDleto/Manifest.toml`


In [5]:

icecat_data = JSON.parsefile("wdc_data_full.json")
df = DataFrame(icecat_data)

1×16 DataFrame
 Row │ CategoryID                         CategoryName                       c ⋯
     │ Dict…                              Dict…                              D ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ Dict{String, Any}("2243"=>"375",…  Dict{String, Any}("2243"=>"Toner…  D ⋯
                                                              14 columns omitted

In [6]:
# Install ITensors if needed
import Pkg; Pkg.add("ITensors")
using ITensors

   Resolving package versions...
      Compat entries added for ITensors
     Project No packages added to or removed from `~/CODE/OpenDleto/Project.toml`
    Manifest No packages added to or removed from `~/CODE/OpenDleto/Manifest.toml`


In [7]:
# Example: Build a 3D tensor from JSON with one-hot encoding
# Assuming your JSON has entries with 3 categorical fields

# Get unique categories for each dimension
function get_categories(data, field)
    unique([entry[field] for entry in data if haskey(entry, field)])
end

# Create index mappings (category -> integer position)
function create_category_map(categories)
    Dict(cat => i for (i, cat) in enumerate(categories))
end

# Build the tensor with one-hot encoding
function build_tensor_from_json(data, field1, field2, field3)
    # Get unique categories for each dimension
    cats1 = get_categories(data, field1)
    cats2 = get_categories(data, field2)
    cats3 = get_categories(data, field3)
    
    # Create mappings
    map1 = create_category_map(cats1)
    map2 = create_category_map(cats2)
    map3 = create_category_map(cats3)
    
    # Create ITensor indices with labels
    idx1 = Index(length(cats1), field1)
    idx2 = Index(length(cats2), field2)
    idx3 = Index(length(cats3), field3)
    
    # Initialize tensor
    T = ITensor(idx1, idx2, idx3)
    
    # Fill tensor with counts (one-hot encoding)
    for entry in data
        if haskey(entry, field1) && haskey(entry, field2) && haskey(entry, field3)
            i = map1[entry[field1]]
            j = map2[entry[field2]]
            k = map3[entry[field3]]
            T[idx1=>i, idx2=>j, idx3=>k] += 1  # Increment count
        end
    end
    
    return T, (cats1, cats2, cats3), (idx1, idx2, idx3)
end

# Print function to display category labels
function show_tensor_info(T, categories, indices)
    println("Tensor dimensions:")
    for (i, (cats, idx)) in enumerate(zip(categories, indices))
        println("  Dim $i ($(ITensors.tags(idx))): $(length(cats)) categories")
        println("    Categories: ", cats[1:min(5, length(cats))], length(cats) > 5 ? "..." : "")
    end
    println("\nNon-zero entries: ", count(x -> x != 0, Array(T, indices...)))
end

show_tensor_info (generic function with 1 method)

In [9]:
# Example usage with your data
# Replace "field1", "field2", "field3" with actual field names from your JSON

# First, inspect the structure of the loaded data
println("Type of icecat_data: ", typeof(icecat_data))
println("\nTop-level keys:")
println(keys(icecat_data))

# If it's a dict with an array, inspect the first entry
if haskey(icecat_data, "data") && !isempty(icecat_data["data"])
    println("\nAvailable fields in first data entry:")
    println(keys(icecat_data["data"][1]))
elseif isa(icecat_data, Vector) && !isempty(icecat_data)
    println("\nAvailable fields in first entry:")
    println(keys(icecat_data[1]))
end

Type of icecat_data: Dict{String, Any}

Top-level keys:
["parent_NodeID", "relationToParent", "CategoryName", "gtin", "tokenmatch", "CategoryID", "identifiers", "url", "cluster_id", "schema.org_properties", "pathlist_ids", "pathlist_names", "nodeID", "title", "parent_schema.org_properties", "desc"]


In [10]:
# Updated functions for dict-based JSON structure
function get_categories_from_dict(data_dict, field)
    if haskey(data_dict, field)
        values = data_dict[field]
        if isa(values, Vector)
            return unique(filter(x -> !isnothing(x) && x != "", values))
        else
            return [values]
        end
    end
    return []
end

function build_tensor_from_dict(data_dict, field1, field2, field3)
    # Get values for each field
    vals1 = haskey(data_dict, field1) ? data_dict[field1] : []
    vals2 = haskey(data_dict, field2) ? data_dict[field2] : []
    vals3 = haskey(data_dict, field3) ? data_dict[field3] : []
    
    # Convert to vectors if needed
    vals1 = isa(vals1, Vector) ? vals1 : [vals1]
    vals2 = isa(vals2, Vector) ? vals2 : [vals2]
    vals3 = isa(vals3, Vector) ? vals3 : [vals3]
    
    # Get unique categories
    cats1 = unique(filter(x -> !isnothing(x) && x != "", vals1))
    cats2 = unique(filter(x -> !isnothing(x) && x != "", vals2))
    cats3 = unique(filter(x -> !isnothing(x) && x != "", vals3))
    
    # Create mappings
    map1 = Dict(cat => i for (i, cat) in enumerate(cats1))
    map2 = Dict(cat => i for (i, cat) in enumerate(cats2))
    map3 = Dict(cat => i for (i, cat) in enumerate(cats3))
    
    # Create ITensor indices with labels
    idx1 = Index(length(cats1), field1)
    idx2 = Index(length(cats2), field2)
    idx3 = Index(length(cats3), field3)
    
    # Initialize tensor
    T = ITensor(idx1, idx2, idx3)
    
    # Fill tensor - each combination of categories gets value 1
    for v1 in vals1, v2 in vals2, v3 in vals3
        if v1 != "" && v2 != "" && v3 != "" && !isnothing(v1) && !isnothing(v2) && !isnothing(v3)
            i = map1[v1]
            j = map2[v2]
            k = map3[v3]
            T[idx1=>i, idx2=>j, idx3=>k] += 1
        end
    end
    
    return T, (cats1, cats2, cats3), (idx1, idx2, idx3)
end

build_tensor_from_dict (generic function with 1 method)

In [11]:
# Example: Build tensor using CategoryName, relationToParent, and parent_NodeID
T, categories, indices = build_tensor_from_dict(icecat_data, "CategoryName", "relationToParent", "parent_NodeID")

show_tensor_info(T, categories, indices)

Tensor dimensions:
  Dim 1 ("CategoryName"): 1 categories
    Categories: Dict{String, Any}[Dict("2243" => "Toner Cartridges", "1881" => "Serial Cables", "2923" => "Internal Hard Drives", "9571" => "Motherboards", "7413" => "Ink Cartridges", "17846" => "Processors", "3215" => "Computer Monitors", "4088" => "Ink Cartridges", "6553" => "Computer Cooling Components", "8861" => "Keyboards", "3157" => "USB Cables", "7192" => "POS Printers", "19031" => "TVs", "7064" => "Ink Cartridges", "8359" => "AV Extenders", "1429" => "Ink Cartridges", "112" => "Ink Cartridges", "5094" => "Computer Cooling Components", "8603" => "Internal Hard Drives", "17" => "Optical Disc Drives", "11446" => "Printer Labels", "11107" => "Notebooks", "6336" => "Computer Monitors", "14463" => "Printer Labels", "11933" => "Memory Modules", "341" => "Mice", "664" => "Motherboards", "12932" => "Ink Cartridges", "4866" => "Camera Lenses", "14" => "Processors", "5465" => "Camera Lenses", "7262" => "Notebooks", "2585" => "Radi

Excessive output truncated after 524288 bytes.

4-atx-motherboard-mb-54r-gi.html>", "18686" => "<http://www.labelzone.co.uk/s0720970-24mm-black-on-red-dymo-d1-tape/p406>", "3204" => "<https://www.overclockers.co.uk/asus-geforce-gtx-1060-dual-oc-6144mb-gddr5-pci-express-graphics-card-gx-40i-as.html>", "14768" => "<https://www.overclockers.co.uk/cooler-master-nepton-240m-aio-cooling-solution-hs-065-cm.html>", "4380" => "<http://www.inkredible.co.uk/epson-expression-premium-xp-800-ink-cartridges-2>", "2639" => "<http://www.inkredible.co.uk/canon-pixma-mg2450-ink-cartridges>", "14866" => "<http://www.inkredible.co.uk/canon-pixma-ts9055-ink-cartridges>", "2956" => "<http://www.jkinformatique.com/contents/fr/d1209.html>", "6526" => "<https://www.overclockers.co.uk/gigabyte-ga-990fx-gaming-amd-990fx-socket-am3-ddr3-atx-motherboard-mb-543-gi.html>", "7949" => "<http://www.inkredible.co.uk/canon-pixma-mp150-ink-cartridges-2>", "9380" => "<https://www.overclockers.co.uk/aerocool-cyclops-advance-midi-tower-gaming-case-red-ca-150-ae.html>", "10

In [ ]:
# Build the tensor (replace with your actual field names)
# Example: T, categories, indices = build_tensor_from_json(icecat_data, "brand", "category", "color")
# 
# show_tensor_info(T, categories, indices)
# 
# Access tensor elements:
# T[indices[1]=>1, indices[2]=>2, indices[3]=>3]  # Access specific element
# 
# The ITensor stores labeled indices, so you can contract with other ITensors
# or perform operations while preserving the semantic meaning of each dimension